In [ ]:
from glob import glob
import stanza
import pandas as pd
# stanza.download("pt")
nlp = stanza.Pipeline("pt", download_method=None)

2026-06-05 10:49:35 INFO: Loading these models for language: pt (Portuguese):
| Processor    | Package         |
----------------------------------
| tokenize     | bosque          |
| mwt          | bosque          |
| pos          | bosque_charlm   |
| lemma        | bosque_nocharlm |
| constituency | cintil_charlm   |
| depparse     | bosque_charlm   |

2026-06-05 10:49:35 INFO: Using device: cuda
2026-06-05 10:49:35 INFO: Loading: tokenize
2026-06-05 10:49:36 INFO: Loading: mwt
2026-06-05 10:49:36 INFO: Loading: pos
2026-06-05 10:49:39 INFO: Loading: lemma
2026-06-05 10:49:40 INFO: Loading: constituency
2026-06-05 10:49:40 INFO: Loading: depparse
2026-06-05 10:49:41 INFO: Done loading processors!


In [ ]:
def process_verb(verb, verbs_df):    
    new_row = {
        "text": verb.text,
        "lemma": verb.lemma
    }

    if verb.feats is None:
        return verbs_df
    
    feats_dict = dict(feat.split("=") for feat in verb.feats.split("|"))

    new_row = new_row | feats_dict
    df_row = pd.DataFrame([new_row])

    verbs_df = pd.concat([verbs_df, df_row], ignore_index=True)

    return verbs_df

In [ ]:
def process_contraction(contraction, contraction_df):
    new_row = {
        "text": contraction.text
    }

    for i, word in enumerate(contraction.words):
        new_row[f"text{i+1}"] = word.text
        new_row[f"lemma{i+1}"] = word.lemma
    
    df_row = pd.DataFrame([new_row])
    contraction_df = pd.concat([contraction_df, df_row], ignore_index=True)

    return contraction_df

In [ ]:
def extract_text_info(text, contraction_df=None, verbs_df=None):
    if contraction_df is None:
        contraction_df = pd.DataFrame()
    if verbs_df is None:
        verbs_df = pd.DataFrame()

    doc = nlp(text)
    print(text)
    for sent in doc.sentences:
        for token in sent.tokens:
            # Check if word is a contraction
            if len(token.words) > 1:
                contraction_df = process_contraction(token, contraction_df)
            else:
                word = token.words[0]
                if word.pos == "VERB":
                    verbs_df = process_verb(word, verbs_df)

    contraction_df.drop_duplicates(inplace=True)
    verbs_df.drop_duplicates(inplace=True)

    return contraction_df, verbs_df


In [ ]:
def try_reading_dataframe(file_name):
    try:
        return pd.read_csv(file_name)
    except:
        return None

In [ ]:
def process_corpus(file_paths):
    contraction_df = try_reading_dataframe("dataframes/contraction.csv")
    verbs_df = try_reading_dataframe("dataframes/verbs.csv")
    
    for i, file_path in enumerate(file_paths):
        
        print(f"File {i+1}/{len(file_paths)} - {file_path}")

        with open(file_path, "r") as file:
            for line in file:
                line = line.strip()
                contraction_df, verbs_df = extract_text_info(line, contraction_df, verbs_df)
    
        contraction_df.to_csv("dataframes/contraction.csv", index=False)
        verbs_df.to_csv("dataframes/verbs.csv", index=False)

    return contraction_df, verbs_df

In [14]:
corpora = glob("../../../corpora/Universal Dependencies/*/*.txt", recursive=True)
corpora = [x for x in corpora if 'LICENSE' not in x]
corpora

['../../../corpora/Universal Dependencies/UD_Portuguese-Bosque/pt_bosque-ud-dev.txt',
 '../../../corpora/Universal Dependencies/UD_Portuguese-Bosque/pt_bosque-ud-train.txt',
 '../../../corpora/Universal Dependencies/UD_Portuguese-Bosque/pt_bosque-ud-test.txt',
 '../../../corpora/Universal Dependencies/UD_Portuguese-GSD/pt_gsd-ud-test.txt',
 '../../../corpora/Universal Dependencies/UD_Portuguese-GSD/pt_gsd-ud-dev.txt',
 '../../../corpora/Universal Dependencies/UD_Portuguese-GSD/pt_gsd-ud-train.txt',
 '../../../corpora/Universal Dependencies/UD_Portuguese-CINTIL/pt_cintil-ud-train.txt',
 '../../../corpora/Universal Dependencies/UD_Portuguese-CINTIL/pt_cintil-ud-test.txt',
 '../../../corpora/Universal Dependencies/UD_Portuguese-CINTIL/pt_cintil-ud-dev.txt',
 '../../../corpora/Universal Dependencies/UD_Portuguese-PetroGold/pt_petrogold-ud-dev.txt',
 '../../../corpora/Universal Dependencies/UD_Portuguese-PetroGold/pt_petrogold-ud-train.txt',
 '../../../corpora/Universal Dependencies/UD_Port

In [ ]:
process_corpus(corpora)